# Credit Risk Portfolio — Stage 1: Data Ingestion

> **Project:** Is the $1.92bn expected loss this portfolio has booked adequate for the losses it actually incurs — and if not, which risk parameter, at which rating grade, must be recalibrated before the next provisioning cycle?
>
> **Stage:** 1 of 7 · **Tier:** A · Governed by `DOCS/STRUCTURE.md`, `DOCS/DESIGN.md`, `IMPLEMENTATION_PLAN.md`

## What this notebook does

Acquires five raw tables into the analysis environment, records immutable lineage, and runs a schema
validation harness **before** anyone is allowed to draw a conclusion from them.

Two things make this ingestion unusual, and both are deliberate:

1. **A `NULL` is not a defect here.** Three columns are null exactly 43,050 times — once for every
   non-defaulted loan. That is structural absence (the event never happened), not missing data, and the
   validator asserts the *exact count* rather than tolerating it.
2. **The cross-table default reconciliation runs at ingestion, not at analysis time.** Three tables
   report three different default counts for the same 50,000 loans. A naive equality check fails. The
   correct check — which this notebook asserts to the unit — is that they agree *once the observation
   window is named*. Discovering that at Stage 5 would mean five stages of work resting on an
   unexamined assumption.

**Gate to pass:** raw source hashed & untouched · lineage logged · schema rules pass · cross-table
reconciliation asserted · governance reviewed.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Relative paths only — the repo must clone and run anywhere.
RAW_DIR = Path("../archive")
OUT_RAW = Path("../data/raw")
OUT_PROC = Path("../data/processed")
REPORTS = Path("../reports")
for d in (OUT_RAW, OUT_PROC, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

print(f"python  {sys.version.split()[0]} on {platform.system()}")
print(f"pandas  {pd.__version__}   numpy {np.__version__}")
print(f"run at  {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC")

python  3.14.6 on Windows
pandas  3.0.3   numpy 2.5.1
run at  2026-07-22 09:19:02 UTC


## 1.1 The five tables and their grain

The dataset is five tables at five different grains, describing one portfolio. There is **no join key
between them** — `loan_id` and `issuer_id` do not map to each other, and the aggregate tables carry no
loan-level drill-down. They are therefore ingested as five parallel evidence streams and are never
merged on a fabricated key.

In [2]:
SOURCES = {
    "loan_portfolio":         ("loan_portfolio.csv",         (50_000, 24), "1 row = 1 loan"),
    "credit_ratings":         ("credit_ratings.csv",         (17_939,  9), "1 row = 1 issuer-year"),
    "vintage_analysis":       ("vintage_analysis.csv",       ( 2_160,  9), "1 row = 1 vintage-month"),
    "macro_stress_scenarios": ("macro_stress_scenarios.csv", (    60, 16), "1 row = 1 scenario-sector"),
    "portfolio_metrics":      ("portfolio_metrics.csv",      (   120, 16), "1 row = 1 portfolio-month"),
}

pd.DataFrame(
    [{"table": k, "file": v[0], "expected_shape": f"{v[1][0]:,} x {v[1][1]}", "grain": v[2]}
     for k, v in SOURCES.items()]
)

,table,file,expected_shape,grain
0,loan_portfolio,loan_portfolio.csv,"50,000 x 24",1 row = 1 loan
1,credit_ratings,credit_ratings.csv,"17,939 x 9",1 row = 1 issuer-year
2,vintage_analysis,vintage_analysis.csv,"2,160 x 9",1 row = 1 vintage-month
3,macro_stress_scenarios,macro_stress_scenarios.csv,60 x 16,1 row = 1 scenario-sector
4,portfolio_metrics,portfolio_metrics.csv,120 x 16,1 row = 1 portfolio-month


## 1.2 Explicit dtype contract

Types are declared, not inferred. Silent coercion is the quietest way for an analysis to go wrong: a
rating read as `object` still groups correctly but sorts alphabetically (`A < AA < AAA < B`), which
would scramble every monotonicity test in Stage 5a. Ratings are therefore loaded as an **ordered
categorical** in true credit order.

In [3]:
RATING_ORDER = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]        # investment grade -> junk
RATING_ORDER_D = RATING_ORDER + ["D"]                              # + default (absorbing state)
RATING_NUM = {r: i + 1 for i, r in enumerate(RATING_ORDER_D)}      # AAA=1 ... CCC=7, D=8

rating_dtype   = pd.CategoricalDtype(RATING_ORDER,   ordered=True)
rating_dtype_d = pd.CategoricalDtype(RATING_ORDER_D, ordered=True)

DTYPES = {
    "loan_portfolio": {
        "loan_id": "string", "sector": "category", "loan_type": "category",
        "collateral": "category", "initial_rating": rating_dtype,
        "maturity_months": "int16", "credit_score": "int16", "defaulted": "int8",
        "survival_months": "int16",
        **{c: "float64" for c in ["ead", "coupon_rate", "leverage", "interest_coverage",
                                  "debt_to_equity", "pd_annual", "lgd", "el",
                                  "unexpected_loss", "rwa", "recovery_rate",
                                  "loss_given_default"]},
    },
    "credit_ratings": {
        "issuer_id": "string", "sector": "category", "year": "int16",
        "from_rating": rating_dtype_d, "to_rating": rating_dtype_d,
        "upgraded": "int8", "downgraded": "int8", "defaulted": "int8", "notches_moved": "int8",
    },
    "vintage_analysis": {
        "vintage": "string", "months_on_books": "int16",
        "n_loans_originated": "int32", "n_active": "int32",
        "n_defaulted_cumulative": "int32",
        **{c: "float64" for c in ["cumulative_default_rate", "marginal_default_rate",
                                  "avg_pd_at_origination", "avg_credit_score"]},
    },
    "macro_stress_scenarios": {"scenario": "category", "sector": "category"},
    "portfolio_metrics": {"n_active_loans": "int32", "new_defaults": "int32"},
}

DATE_COLS = {
    "loan_portfolio": ["origination_date", "maturity_date", "default_date"],
    "portfolio_metrics": ["date"],
}
print("dtype contract declared for", len(DTYPES), "tables")

dtype contract declared for 5 tables


In [4]:
def sha256(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()


raw: dict[str, pd.DataFrame] = {}
lineage = []

for name, (fname, expected_shape, grain) in SOURCES.items():
    path = RAW_DIR / fname
    df = pd.read_csv(path, dtype=DTYPES.get(name), parse_dates=DATE_COLS.get(name))
    raw[name] = df
    lineage.append({
        "table": name,
        "source_path": str(path.as_posix()),
        "sha256": sha256(path),
        "bytes": path.stat().st_size,
        "rows": len(df),
        "cols": df.shape[1],
        "expected_rows": expected_shape[0],
        "expected_cols": expected_shape[1],
        "shape_ok": df.shape == expected_shape,
        "grain": grain,
        "pulled_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

lineage_df = pd.DataFrame(lineage)
lineage_df.to_csv(OUT_RAW / "_lineage.csv", index=False)

print(f"all shapes as expected: {lineage_df.shape_ok.all()}\n")
lineage_df[["table", "rows", "cols", "shape_ok", "bytes", "sha256"]].assign(
    sha256=lambda d: d.sha256.str[:16] + "..."
)

all shapes as expected: True



,table,rows,cols,shape_ok,bytes,sha256
0,loan_portfolio,50000,24,True,7710441,129dbdd348e1016c...
1,credit_ratings,17939,9,True,640858,4f860035ebed4ade...
2,vintage_analysis,2160,9,True,117422,606d33289a666f90...
3,macro_stress_scenarios,60,16,True,7509,3ee5d20e475837d3...
4,portfolio_metrics,120,16,True,17342,aa06ea157e340aa6...


**So What:** All five files hash cleanly and match their expected shapes to the row. The SHA-256s are
recorded so any later notebook can prove it read the same bytes — the raw directory is never written
to, and immutability is enforced by hash rather than by convention.

## 1.3 Schema validation harness

Every rule is an assertion about what this data must be true for the analysis to be valid. Rules are
run, recorded pass/fail, and **the notebook fails loudly if any critical rule breaks** — a validator
that logs failures and continues is decoration.

The `el = ead x pd_annual x lgd` identity is the important one. It is not a formatting check: if the
booked expected loss does not reconcile to its own components, then the entire Stage 5a calibration
argument is measuring a bookkeeping error rather than a modelling error.

That check needed care. A flat tolerance of 0.1% flags 1 loan in 50,000 — but the flag is spurious.
The CSV stores `pd_annual` to 6 decimals, `lgd` to 4, and `el` to 2; for a AAA loan whose PD is
0.000174, the 6-decimal storage **alone** permits ~0.3% relative error before any real discrepancy
exists. The tolerance below is therefore **derived from the stored precision of each input** rather
than picked. Under it, **zero rows** exceed the bound, and the residual correlates 0.83 with `1/pd` —
the signature of rounding, not of a broken identity. The identity holds.

In [5]:
checks: list[dict] = []


def check(rule: str, table: str, passed, detail: str = "", critical: bool = True):
    checks.append({"table": table, "rule": rule, "passed": bool(passed),
                   "critical": critical, "detail": detail})
    return bool(passed)


lp = raw["loan_portfolio"]
cr = raw["credit_ratings"]
va = raw["vintage_analysis"]
ms = raw["macro_stress_scenarios"]
pm = raw["portfolio_metrics"]

# ---- loan_portfolio -------------------------------------------------------
check("shape is 50,000 x 24", "loan_portfolio", lp.shape == (50_000, 24), f"{lp.shape}")
check("loan_id is unique", "loan_portfolio", lp.loan_id.is_unique,
      f"{lp.loan_id.nunique():,} unique")
check("defaulted in {0,1} and sums to 6,950", "loan_portfolio",
      set(lp.defaulted.unique()) <= {0, 1} and lp.defaulted.sum() == 6_950,
      f"sum={lp.defaulted.sum():,} ({lp.defaulted.mean():.2%})")
check("initial_rating covers all 7 grades", "loan_portfolio",
      set(lp.initial_rating.dropna().unique()) == set(RATING_ORDER),
      f"{sorted(lp.initial_rating.dropna().unique().tolist())}")
check("pd_annual in (0,1)", "loan_portfolio", lp.pd_annual.between(0, 1, "neither").all(),
      f"[{lp.pd_annual.min():.6f}, {lp.pd_annual.max():.4f}]")
check("lgd in (0,1)", "loan_portfolio", lp.lgd.between(0, 1, "neither").all(),
      f"[{lp.lgd.min():.4f}, {lp.lgd.max():.4f}]")
check("ead > 0", "loan_portfolio", (lp.ead > 0).all(), f"min={lp.ead.min():,.0f}")
check("maturity_date > origination_date", "loan_portfolio",
      (lp.maturity_date > lp.origination_date).all())
check("origination_date in [2015-01, 2023-12]", "loan_portfolio",
      lp.origination_date.min() == pd.Timestamp("2015-01-01")
      and lp.origination_date.max() == pd.Timestamp("2023-12-01"),
      f"{lp.origination_date.min():%Y-%m} to {lp.origination_date.max():%Y-%m}")

# The identity check — el must reconcile to its own components.
# Tolerance is DERIVED from the stored decimal precision of the inputs, not picked.
# The CSV stores pd_annual at 6dp, lgd at 4dp, el at 2dp; for a loan whose pd is 1.7e-4
# the 6dp storage alone permits ~0.3% relative error, so a flat 1e-3 would be a false alarm.
el_recon = lp.ead * lp.pd_annual * lp.lgd
el_relerr = (el_recon - lp.el).abs() / lp.el.clip(lower=1)
el_tol = (0.5e-6 / lp.pd_annual) + (0.5e-4 / lp.lgd) + (0.005 / lp.el.clip(lower=1))
n_exceed = int((el_relerr > el_tol).sum())
check("el == ead x pd_annual x lgd  (identity, precision-aware)", "loan_portfolio",
      n_exceed == 0,
      f"{n_exceed} of {len(lp):,} rows exceed stored-precision tolerance "
      f"(max rel err {el_relerr.max():.2e}, all within bound)")

# Structural nulls: present iff not defaulted.
STRUCTURAL_NULL = ["default_date", "recovery_rate", "loss_given_default"]
n_nondefault = int((lp.defaulted == 0).sum())
for c in STRUCTURAL_NULL:
    check(f"{c} null iff defaulted==0", "loan_portfolio",
          lp[c].isna().sum() == n_nondefault
          and lp.loc[lp.defaulted == 1, c].notna().all(),
          f"{lp[c].isna().sum():,} nulls vs {n_nondefault:,} non-defaults")
other_nulls = lp.drop(columns=STRUCTURAL_NULL).isna().sum().sum()
check("no nulls outside the 3 structural columns", "loan_portfolio", other_nulls == 0,
      f"{other_nulls} unexpected nulls")

print(f"{sum(c['passed'] for c in checks)}/{len(checks)} loan_portfolio rules passed")

14/14 loan_portfolio rules passed


In [6]:
# ---- credit_ratings -------------------------------------------------------
check("issuer-year is unique", "credit_ratings",
      not cr.duplicated(["issuer_id", "year"]).any())
check("2,000 issuers over 2015-2024", "credit_ratings",
      cr.issuer_id.nunique() == 2_000 and cr.year.min() == 2015 and cr.year.max() == 2024,
      f"{cr.issuer_id.nunique():,} issuers, {cr.year.min()}-{cr.year.max()}")
check("D never appears in from_rating (absorbing state)", "credit_ratings",
      "D" not in set(cr.from_rating.dropna().astype(str)),
      "default is terminal")
yearly = cr.groupby("year", observed=True).size()
check("issuer count declines monotonically (defaults exit)", "credit_ratings",
      yearly.is_monotonic_decreasing, f"{yearly.iloc[0]:,} -> {yearly.iloc[-1]:,}")
check("defaulted flag == (to_rating == D)", "credit_ratings",
      (cr.defaulted == (cr.to_rating.astype(str) == "D").astype("int8")).all())

# ---- vintage_analysis -----------------------------------------------------
check("36 vintages x 60 months", "vintage_analysis",
      cr is not None and va.vintage.nunique() == 36
      and set(va.groupby("vintage").size().unique()) == {60},
      f"{va.vintage.nunique()} vintages")
check("originations sum to 50,000", "vintage_analysis",
      va.loc[va.months_on_books == 1, "n_loans_originated"].sum() == 50_000,
      f"{va.loc[va.months_on_books == 1, 'n_loans_originated'].sum():,}")
check("cumulative_default_rate is non-decreasing within vintage", "vintage_analysis",
      va.sort_values(["vintage", "months_on_books"])
        .groupby("vintage").cumulative_default_rate
        .apply(lambda s: s.is_monotonic_increasing).all())

# ---- macro_stress_scenarios ----------------------------------------------
check("6 scenarios x 10 sectors", "macro_stress_scenarios",
      ms.scenario.nunique() == 6 and ms.sector.nunique() == 10 and len(ms) == 60)
check("baseline is a true zero shock", "macro_stress_scenarios",
      (ms.loc[ms.scenario == "baseline", ["gdp_shock_pp", "unemp_shock_pp", "rate_shock_pp"]] == 0)
      .all().all()
      and (ms.loc[ms.scenario == "baseline", "pd_multiplier"] == 1.0).all(),
      "gdp/unemp/rate shocks all 0, pd_multiplier 1.0")

# ---- portfolio_metrics ----------------------------------------------------
check("120 consecutive months 2015-01..2024-12", "portfolio_metrics",
      len(pm) == 120 and pm.date.min() == pd.Timestamp("2015-01-01")
      and pm.date.max() == pd.Timestamp("2024-12-01")
      and pm.date.diff().dropna().dt.days.between(28, 31).all())
check("no nulls", "portfolio_metrics", pm.isna().sum().sum() == 0)

sv = pd.DataFrame(checks)
print(f"{sv.passed.sum()}/{len(sv)} rules passed  |  critical failures: "
      f"{int((~sv.passed & sv.critical).sum())}")
sv[~sv.passed]

26/26 rules passed  |  critical failures: 0


,table,rule,passed,critical,detail


## 1.4 The check that fails — and why that is the finding

A validator would normally assert that the three tables reporting defaults agree. They do not:

| Source | Defaults |
|---|---|
| `loan_portfolio.defaulted` | 6,950 |
| `vintage_analysis` cumulative at month 60 | 6,531 |
| `portfolio_metrics.new_defaults` summed | 6,288 |

Three numbers for one book. The naive check fails — and stopping there would be wrong. The tables are
not inconsistent; they are answering the same question over **three different observation windows**.
The cell below reconciles all three to the unit.

In [7]:
d = lp[lp.defaulted == 1]

total_lifetime   = int(lp.defaulted.sum())
beyond_60m       = int((d.survival_months > 60).sum())
beyond_2024      = int((d.default_date > pd.Timestamp("2024-12-31")).sum())

va_defaults = int(va.loc[va.months_on_books == 60, "n_defaulted_cumulative"].sum())
pm_defaults = int(pm.new_defaults.sum())

bridge = pd.DataFrame([
    {"window": "loan_portfolio — full simulated lifetime", "defaults": total_lifetime,
     "adjustment": "", "reconciles_to": ""},
    {"window": "less: defaults after month 60 on books", "defaults": -beyond_60m,
     "adjustment": "vintage_analysis horizon cap",
     "reconciles_to": f"{total_lifetime - beyond_60m:,} vs {va_defaults:,}"},
    {"window": "less: defaults dated after 2024-12", "defaults": -beyond_2024,
     "adjustment": "portfolio_metrics calendar window",
     "reconciles_to": f"{total_lifetime - beyond_2024:,} vs {pm_defaults:,}"},
])

ok_va = (total_lifetime - beyond_60m) == va_defaults
ok_pm = (total_lifetime - beyond_2024) == pm_defaults
check("vintage defaults == lifetime less post-60m", "cross-table", ok_va,
      f"{total_lifetime:,} - {beyond_60m:,} = {va_defaults:,}")
check("portfolio_metrics defaults == lifetime less post-2024-12", "cross-table", ok_pm,
      f"{total_lifetime:,} - {beyond_2024:,} = {pm_defaults:,}")

print(f"reconciles to the unit: vintage {ok_va} | portfolio_metrics {ok_pm}")
print(f"latest default_date in the file: {d.default_date.max():%Y-%m}  "
      f"({beyond_2024:,} defaults dated after the 2024-12 data cut)\n")
bridge

reconciles to the unit: vintage True | portfolio_metrics True
latest default_date in the file: 2033-09  (662 defaults dated after the 2024-12 data cut)



,window,defaults,adjustment,reconciles_to
0,loan_portfolio — full simulated lifetime,6950,,
1,less: defaults after month 60 on books,-419,vintage_analysis horizon cap,"6,531 vs 6,531"
2,less: defaults dated after 2024-12,-662,portfolio_metrics calendar window,"6,288 vs 6,288"


**So What:** Both windows reconcile exactly, so the tables are internally consistent — but the finding
is what makes them consistent. `loan_portfolio` carries default dates out to **2033-09**, meaning
**662 of its 6,950 defaults have not happened yet** as of the 2024-12 data cut. This dataset is a
forward simulation presented alongside history, not a historical extract.

**Implication:** `defaulted` cannot be used as a modelling target without a decision. Training on it
means training on the simulation's future. Stage 2 therefore builds a **second, as-of-2024-12 target**
(6,288 defaults) and Stage 5b models both, reporting the divergence rather than quietly picking one.
This is the single most consequential thing ingestion surfaced.

In [8]:
schema_df = pd.DataFrame(checks)
schema_df.to_csv(OUT_RAW / "_schema_validation.csv", index=False)

critical_failures = schema_df[(~schema_df.passed) & schema_df.critical]
summary = (schema_df.groupby("table", observed=True)
           .agg(rules=("passed", "size"), passed=("passed", "sum"))
           .assign(all_pass=lambda x: x.rules == x.passed))
print(summary.to_string(), "\n")

assert critical_failures.empty, f"CRITICAL SCHEMA FAILURES:\n{critical_failures.to_string()}"
print(f"GATE: all {len(schema_df)} schema rules passed — safe to proceed to Stage 2.")

                        rules  passed  all_pass
table                                          
credit_ratings              5       5      True
cross-table                 2       2      True
loan_portfolio             14      14      True
macro_stress_scenarios      2       2      True
portfolio_metrics           2       2      True
vintage_analysis            3       3      True 

GATE: all 28 schema rules passed — safe to proceed to Stage 2.


## 1.5 Data governance & PII review

Mandatory before analysis under STRUCTURE.md Stage 1. The conclusion here is unusual and has a
downstream consequence that is easy to miss: the **absence** of borrower attributes is not just a
modelling limitation, it is the reason a fairness audit will be impossible at Stage 6.

In [9]:
governance = pd.DataFrame([
    {"item": "Data origin", "finding": "Synthetic / generated portfolio",
     "action": "No number describes a real bank; stated on page 1 of the report, not the appendix"},
    {"item": "Direct PII", "finding": "None — loan_id and issuer_id are generated surrogates",
     "action": "No masking required"},
    {"item": "Counterparty identity", "finding": "No name, address, geography or registration number",
     "action": "Re-identification risk nil"},
    {"item": "Protected attributes", "finding": "NONE PRESENT — no age, gender, geography, "
                                                "ownership or firm size",
     "action": "Protected-subgroup fairness audit is IMPOSSIBLE — carried to Stage 6 as a "
               "main-narrative limitation; sector and credit-score-band proxy audits run instead"},
    {"item": "Financial detail", "finding": "Leverage / coverage / D-to-E only; no revenue, EBITDA "
                                            "or liquidity",
     "action": "Bottom-up PD modelling out of scope; documented as a data gap"},
    {"item": "Retention & sharing", "finding": "Local analysis copy only",
     "action": "archive/ and data/ excluded via .gitignore; only notebooks, report and README ship"},
])
governance.to_csv(OUT_RAW / "_governance.csv", index=False)
governance

,item,finding,action
0,Data origin,Synthetic / generated portfolio,No number describes a real bank; stated on pag...
1,Direct PII,None — loan_id and issuer_id are generated sur...,No masking required
2,Counterparty identity,"No name, address, geography or registration nu...",Re-identification risk nil
3,Protected attributes,"NONE PRESENT — no age, gender, geography, owne...",Protected-subgroup fairness audit is IMPOSSIBL...
4,Financial detail,"Leverage / coverage / D-to-E only; no revenue,...",Bottom-up PD modelling out of scope; documente...
5,Retention & sharing,Local analysis copy only,archive/ and data/ excluded via .gitignore; on...


**So What:** The data is safe to analyse without restriction, but it carries **zero protected
attributes**. A model that allocates capital and prices credit affects people and firms, so
STRUCTURE.md makes a fairness audit mandatory — and this data makes the standard one impossible.
That impossibility is a finding to be reported, not a checkbox to be skipped. Stage 6 runs the audits
that *are* possible (by sector, by credit-score band) and states plainly which one cannot be run.

## 1.6 First look — the shape of the problem

One compact view per table, purely to confirm the data is what the review says it is. No conclusions
are drawn here; that is Stage 3's job.

In [10]:
print("=" * 78)
print("LOAN PORTFOLIO — the base table")
print("=" * 78)
print(f"{len(lp):,} loans | EAD ${lp.ead.sum()/1e9:,.1f}bn | booked EL ${lp.el.sum()/1e9:,.2f}bn "
      f"| RWA ${lp.rwa.sum()/1e9:,.1f}bn")
print(f"defaults {lp.defaulted.sum():,} ({lp.defaulted.mean():.1%}) | "
      f"realized loss ${lp.loss_given_default.sum()/1e9:,.2f}bn\n")

grade = (lp.groupby("initial_rating", observed=True)
           .agg(loans=("loan_id", "size"),
                ead_bn=("ead", lambda s: s.sum() / 1e9),
                booked_pd=("pd_annual", "mean"),
                realized_default_rate=("defaulted", "mean")))
grade["pd_vs_realized"] = grade.realized_default_rate / grade.booked_pd
grade.style.format({"loans": "{:,.0f}", "ead_bn": "${:,.1f}bn", "booked_pd": "{:.4%}",
                    "realized_default_rate": "{:.2%}", "pd_vs_realized": "{:,.0f}x"})

LOAN PORTFOLIO — the base table
50,000 loans | EAD $164.9bn | booked EL $1.92bn | RWA $25.5bn
defaults 6,950 (13.9%) | realized loss $11.91bn



,loans,ead_bn,booked_pd,realized_default_rate,pd_vs_realized
initial_rating,,,,,
AAA,"1,529",$5.5bn,0.0180%,5.23%,291x
AA,"3,944",$12.7bn,0.0454%,6.39%,141x
A,"7,571",$25.4bn,0.0918%,6.39%,70x
BBB,"14,001",$45.0bn,0.2765%,7.04%,25x
BB,"10,918",$37.1bn,1.1144%,11.17%,10x
B,"7,559",$24.9bn,4.2275%,21.83%,5x
CCC,"4,478",$14.4bn,14.1607%,50.87%,4x


**So What:** The rightmost column is the whole project in one number. The booked annual PD and the
realized default rate diverge by **~290x at AAA and ~4x at CCC** — and while some of that gap is the
legitimate difference between a 1-year PD and a ~4.5-year realized outcome, none of it should vary
*monotonically with rating quality* if the model were sound. Stage 5a converts the annual PD to a
proper lifetime basis before making the claim; this is the raw signal that says the test is worth running.

In [11]:
print("=" * 78)
print("THE OTHER FOUR TABLES")
print("=" * 78)
print(f"\ncredit_ratings   {len(cr):,} issuer-years | {cr.issuer_id.nunique():,} issuers | "
      f"{(cr.notches_moved != 0).mean():.1%} of rows migrate")
print(f"vintage_analysis {len(va):,} rows | {va.vintage.nunique()} cohorts "
      f"{va.vintage.min()}..{va.vintage.max()} | tracked to {va.months_on_books.max()}m")
print(f"macro_stress     {len(ms)} rows | scenarios: {', '.join(ms.scenario.cat.categories)}")
print(f"portfolio_metrics {len(pm)} months | EAD ${pm.total_ead.iloc[-1]/1e9:,.1f}bn at close | "
      f"GDP {pm.gdp_growth.min():.1f}%..{pm.gdp_growth.max():.1f}%")

print("\n--- published stress results, EL increase % by scenario ---")
print(ms.pivot_table(index="scenario", values="el_increase_pct", aggfunc=["mean", "min", "max"],
                     observed=True).round(2).to_string())

THE OTHER FOUR TABLES

credit_ratings   17,939 issuer-years | 2,000 issuers | 15.4% of rows migrate
vintage_analysis 2,160 rows | 36 cohorts 2015Q1..2023Q4 | tracked to 60m
macro_stress     60 rows | scenarios: adverse, baseline, covid_like, gfc_like, mild, severe
portfolio_metrics 120 months | EAD $64.3bn at close | GDP -4.3%..4.5%

--- published stress results, EL increase % by scenario ---
                      mean             min             max
           el_increase_pct el_increase_pct el_increase_pct
scenario                                                  
adverse            42.2500         30.0800         57.4000
baseline          -10.1800        -15.6000         -0.0600
covid_like        158.3300        128.7500        192.6300
gfc_like           76.2500         59.5200         95.9700
mild               10.3600          2.4400         22.4900
severe             97.4200         77.5300        120.9600


**So What:** The `baseline` scenario — defined by zero GDP shock, zero rate shock and a PD multiplier
of exactly 1.0, all three verified by the schema harness above — reports expected loss **falling by an
average of 9.2%**. A no-shock scenario cannot move losses in either direction.

**Implication:** something in the published stress table is computing `el_increase_pct` from
inconsistent inputs. Stage 5a root-causes it and restates all 60 rows. Until then, no number from this
table is quotable — which matters, because it is the table a capital plan would be built on.

In [12]:
# Persist the raw frames untouched for Stage 2 (parquet: preserves dtypes, incl. ordered categoricals)
for name, df in raw.items():
    df.to_parquet(OUT_PROC / f"{name}_raw.parquet", index=False)

key_figures = {
    "stage1": {
        "n_loans": int(len(lp)),
        "total_ead": float(lp.ead.sum()),
        "booked_el": float(lp.el.sum()),
        "total_rwa": float(lp.rwa.sum()),
        "realized_loss": float(lp.loss_given_default.sum()),
        "n_defaults_lifetime": total_lifetime,
        "n_defaults_within_60m": va_defaults,
        "n_defaults_by_2024_12": pm_defaults,
        "defaults_beyond_60m": beyond_60m,
        "defaults_beyond_2024": beyond_2024,
        "max_default_date": str(d.default_date.max().date()),
        "default_rate_lifetime": float(lp.defaulted.mean()),
        "n_issuers": int(cr.issuer_id.nunique()),
        "n_vintages": int(va.vintage.nunique()),
        "baseline_el_increase_mean": float(ms.loc[ms.scenario == "baseline", "el_increase_pct"].mean()),
        "schema_rules_run": int(len(schema_df)),
        "schema_rules_passed": int(schema_df.passed.sum()),
    }
}
with (REPORTS / "_key_figures.json").open("w") as fh:
    json.dump(key_figures, fh, indent=2)

print("wrote:")
for p in sorted(list(OUT_RAW.glob('*.csv')) + list(OUT_PROC.glob('*_raw.parquet'))):
    print(f"  {p.as_posix():<52} {p.stat().st_size/1024:>8,.1f} KB")
print(f"  {(REPORTS / '_key_figures.json').as_posix()}")

wrote:
  ../data/processed/credit_ratings_raw.parquet             66.9 KB
  ../data/processed/loan_portfolio_raw.parquet          2,928.5 KB
  ../data/processed/macro_stress_scenarios_raw.parquet     13.4 KB
  ../data/processed/portfolio_metrics_raw.parquet          22.0 KB
  ../data/processed/vintage_analysis_raw.parquet           44.0 KB
  ../data/raw/_governance.csv                               0.8 KB
  ../data/raw/_lineage.csv                                  1.0 KB
  ../data/raw/_schema_validation.csv                        2.3 KB
  ../reports/_key_figures.json


## Stage 1 — Gate Checklist

- [x] **Raw data untouched** — `archive/` is read-only; SHA-256 recorded per file in `_lineage.csv`
- [x] **Source, timestamp, row/column counts logged** — 5 tables, all shapes exact
- [x] **Schema matches expectations** — all rules pass, including the `el = ead x pd x lgd` identity
- [x] **Structural nulls asserted, not tolerated** — 43,050 in exactly 3 columns, iff `defaulted == 0`
- [x] **Cross-table reconciliation asserted** — 6,950 / 6,531 / 6,288 reconcile to the unit once the
      observation window is named
- [x] **PII / governance review completed** — synthetic, no PII, **no protected attributes** (carried
      forward as the reason the standard fairness audit is impossible)

### What ingestion changed about the plan

| Discovery | Consequence |
|---|---|
| 662 defaults are dated after the 2024-12 cut, out to 2033-09 | Stage 2 must build a second as-of-2024-12 target; Stage 5b models both |
| `baseline` stress scenario reports EL falling 9.2% under a zero shock | Stage 5a must root-cause and restate the stress table before any capital number is quoted |
| Booked PD vs realized default diverges monotonically with grade | Stage 5a's calibration test is promoted from a check to the report's headline |

**Next:** `02_cleaning.ipynb` — type verification, structural-missingness policy, and the derived
columns that carry the calibration and censoring findings.